# Preprocessing Pipeline

Preprocessing pipeline for the analysis of the hyperscanning experiment. 
Cotains the ICA procedure of Stopahl et al. 2018, but implemented in MNE python. 
For now it can process one recording at a time, in the future this may change. 

## Loading the data

Code from build.py

In [ ]:
"""

When the real data is available, can run this in the terminal to get the files

uv run build.py --data-root /mnt/FastData/EXPERIMENT_DATA --out out/
uv run build.py --physics        # adds the physics-state table (~17M rows, 266 MB)
uv run build.py --adhoc-session  # a run that is not in sessions.toml (see Curation)
uv run build.py --export-eeg     # also write each aligned hyperscanner file as .fif
"""

Loading the packages and the raw data

In [ ]:
import mne
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from mne.preprocessing import ICA
%matplotlib inline

In [ ]:
#cell that I will change for each recording for now. Mostly settings to keep track of my defs 
file = Path("out/eeg/raw_eeg.fif")
bad_channels = []
exclude = []  #components to exclude from ICA 

ica_highpass = 1
ica_lowpass = 40 

In [ ]:
#loading cell

raw = mne.io.read_raw_fif(file, preload=True)

print(f"File: {file.name}, Channels: {raw.info['nchan']}, Duration: {raw.times[-1]:.2f} s, Sampling Rate: {raw.info['sfreq']} Hz")
print(raw.ch_names)

#standard 1020 montage
montage = mne.channels.make_standard_montage('standard_1020') #double check with luke (may have own location file)

raw.pick("eeg")  #only keep EEG channels
raw.drop_channels(["M1", "M2"]) #drop mastoids, not used for reference (Anna)

#taking a look at raw data 
raw.plot(n_channels=32, scalings="auto")




## ICA on raw data

**Looping over subjects:**

Filtering the data for ICA training

Epoching data and removing artificial epochs for ICA

Running ICA with the runica algorithm and the additional option of PCA

Saving ICA weights to original data


In [ ]:
#identifying bad channels
#making a copy of the raw data to filter and plot, so i don't modify the original raw data

raw.copy().filter(l_freq=1, h_freq=40).plot(n_channels=32, scalings=dict(eeg=50e-6)) 

#then i update the bad_channels list with the channels that look bad in the plot
bad_channels =[]  


In [ ]:
#for fixing bad channels 
raw.info["bads"] = bad_channels
if bad_channels:
    raw.interpolate_bads()
raw.set_eeg_reference("average") #Anna said to use common average reference

In [ ]:
#doing the ica on a copy 

raw_ica = raw.copy()
raw_ica.filter(l_freq=ica_highpass, h_freq=ica_lowpass)
raw_ica.resample(250) #just to make it faster

n_comp = len(raw.ch_names) - 1 - len(bad_channels)

ica = ICA(n_components=n_comp, method="infomax", fit_params=dict(extended=True)) #stropahl uses infomax, but not sure which one is best for this kind of data
ica.fit(raw_ica, reject=dict(eeg=500e-6), tstep=1.0)

ica.save(file.with_name(file.stem + "-ica.fif"), overwrite=True)

In [ ]:
#investigating the components 
ica.plot_components(inst=raw_ica) #this shows topographies, iewhere on the head each component is
ica.plot_sources(raw_ica) #what each component does over time

In [ ]:
#before vs after for the components in exclude_components
ica.plot_overlay(raw_ica, exclude=exclude_components, picks="eeg")

#remove them from the unfiltered data
raw_clean = ica.apply(raw.copy(), exclude=exclude_components)

#Post-ICA filtering: tbd with Anna (broadband vs alpha/beta)
#no need to remove (unless VERY stereotypical!) heartbeat

raw_clean.save(file.with_name(file.stem + "_clean_raw.fif"), overwrite=True)

In [ ]:
#corrmap
from mne.preprocessing import read_ica, corrmap

ica_files = sorted(Path("out/eeg").glob("*-ica.fif"))
icas = [read_ica(f) for f in ica_files]
for i, f in enumerate(ica_files):
    print(i, f.name)        #which number is which recording (if i can get it to work with multiple)

#need to talk to anna about template??
template_recording = 0      #example
template_component = 0      #example

corrmap(icas, template=(template_recording, template_component),
        threshold="auto", label="eye", plot=True)

#which components matched the template in each recording
for f, ica_i in zip(ica_files, icas):
    print(f.name, "eye components:", ica_i.labels_.get("eye", []))